In [1]:
import pandas as pd

data = pd.read_csv("../data/fd_dataset_messy.csv")
print(data.head(5))

                         email                 time        subject  \
0  mukesh.bhatt@rediffmail.com  2025-02-17 20:29:47  Payment issue   
1      sanjay.jain@hotmail.com  2024-06-07 16:22:21           Help   
2         ashok.nair@gmail.com  2024-04-05 08:16:39     Loan query   
3    manoj.gupta51@hotmail.com  2025-01-04 23:24:05          Query   
4     vinod.chopra87@gmail.com  2025-03-03 10:12:43    Loan status   

                                             content              label  
0  Branch gaya tha, unhone email karne bola. Two ...             Non-FD  
1  Dear customer care, My insurance policy is rea...             Non-FD  
2  Namaskar, Rs 6 lakh kat gaya bina bataye insur...             Non-FD  
3  Dear customer care, 1. Suna hai Bajaj Finance ...  Multiple Category  
4  Dear customer care, Where is my money? The rev...             Non-FD  


##### 1. ONE-HOT ENCODING (binary: 1 if word appears in doc, 0 if not)

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

text = data['content'].astype(str)

onehot_vec = CountVectorizer(binary=True, max_features=20)  # max_features just keeps output readable
onehot_matrix = onehot_vec.fit_transform(text)
onehot_df = pd.DataFrame(onehot_matrix.toarray(), columns=onehot_vec.get_feature_names_out())
print("First Email Sample:", data['content'][0])
print("\n")
print("ONE-HOT ENCODING")
print(onehot_df.head(1))

First Email Sample: Branch gaya tha, unhone email karne bola. Two wheeler loan ki EMI is mahine zyada kati hai. Pehle se Rs 800 jyada. Kyu? Statement bhejo. Bahut pareshan ho gaya hoon. Mukesh Bhatt


ONE-HOT ENCODING
   and  fd  for  gaya  hai  ho  is  it  ka  kya  lakh  loan  my  paisa  sir  \
0    0   0    0     1    1   1   1   0   0    0     0     1   0      0    0   

   the  this  to  want  your  
0    0     0   0     0     0  


Why we can't see Dear in One Hot Encoding Output
- That's expected behavior, not a bug — here's why.
- max_features=20 doesn't pick words based on this one email. 
- It looks at word frequency across the entire corpus (all 2500 rows in data['content']), ranks every word, and keeps only the top 20 most frequent ones. 
- "Dear" appears in this email, but it's not frequent enough across the whole dataset to make that cut.

##### 2. BAG OF WORDS (raw word counts per document)

In [3]:
bow_vec = CountVectorizer(max_features=20)
bow_matrix = bow_vec.fit_transform(text)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_vec.get_feature_names_out())
print("First Email Sample:", data['content'][0])
print("\n")
print("BAG OF WORDS")
print(bow_df.head(1))

First Email Sample: Branch gaya tha, unhone email karne bola. Two wheeler loan ki EMI is mahine zyada kati hai. Pehle se Rs 800 jyada. Kyu? Statement bhejo. Bahut pareshan ho gaya hoon. Mukesh Bhatt


BAG OF WORDS
   and  fd  for  gaya  hai  ho  is  it  ka  kya  lakh  loan  my  paisa  sir  \
0    0   0    0     2    1   1   1   0   0    0     0     1   0      0    0   

   the  this  to  you  your  
0    0     0   0    0     0  


##### 3. TF-IDF (weighted by term frequency × inverse document frequency)


In [4]:
tfidf_vec = TfidfVectorizer(max_features=20)
tfidf_matrix = tfidf_vec.fit_transform(text)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out())
print("First Email Sample:", data['content'][0])
print("\n")
print("TF-IDF")
print(tfidf_df.head(1).round(3))

First Email Sample: Branch gaya tha, unhone email karne bola. Two wheeler loan ki EMI is mahine zyada kati hai. Pehle se Rs 800 jyada. Kyu? Statement bhejo. Bahut pareshan ho gaya hoon. Mukesh Bhatt


TF-IDF
   and   fd  for  gaya    hai     ho     is   it   ka  kya  lakh   loan   my  \
0  0.0  0.0  0.0  0.75  0.272  0.379  0.291  0.0  0.0  0.0   0.0  0.368  0.0   

   paisa  sir  the  this   to  you  your  
0    0.0  0.0  0.0   0.0  0.0  0.0   0.0  


There's real signal in those numbers. Let's break down where each value comes from.
- The mechanics: TF-IDF for a word = (times it appears in this email) × (how rare it is across all 2500 emails), then the whole row gets rescaled (L2-normalized) so its values sum-of-squares to 1. 

Here's the breakdown for the 5 nonzero words in this email:
- Note 1 — gaya (TF-IDF = 0.750, the highest): Appears 2 times in this email ("Branch gaya tha" ... "ho gaya hoon"). Shows up in 613 of 2500 emails (IDF = 2.40). High weight here mainly because it repeats, not because it's especially rare.
- Note 2 — ho (TF-IDF = 0.379): Appears once. Shows up in only 598 of 2500 emails (IDF = 2.43) — the rarest of the five words present, so it gets a decent boost even though it occurs just once.
- Note 3 — loan (TF-IDF = 0.368): Appears once. Shows up in 642 of 2500 emails (IDF = 2.36). This is the most topically meaningful word in the list, and it earns a solid weight.
- Note 4 — is (TF-IDF = 0.291): Appears once. Shows up in 1053 of 2500 emails (IDF = 1.86) — common across the corpus, so it's weighted down despite being present.
= Note 5 — hai (TF-IDF = 0.272, the lowest of the five present)- Appears once. Shows up in 1190 of 2500 emails (IDF = 1.74) — the most common word among the five, so IDF pushes its weight down the most.

Key inferences:
- Zero ≠ unimportant — zero means absent. All 15 zero-columns (and, fd, for, my, the, this, to, you, your, ka, kya, lakh, paisa, sir, it) simply don't appear in this email at all. TF-IDF can't be nonzero for a word that isn't there.
- Repetition matters more than rarity here. "gaya" tops the list mainly because it appears twice, even though its rarity (IDF) is similar to "loan" and "ho."
- IDF is visibly down-weighting common words. "hai" and "is" — the two most globally common words in this set — end up with the lowest weights among the words present, exactly as TF-IDF is designed to do.
- The nonzero words match the email's actual topic. gaya/ho/hai/is/loan all fit an EMI/loan-deduction complaint — and the row's actual label is "Non-FD" (a service complaint, not a fraud case), so the signal is topically consistent.
- Limitation to flag: because max_features=20 picked words by raw frequency, the vocabulary is mostly filler words (hai, is, the, to, my) rather than fraud-distinguishing terms. TF-IDF's real advantage — surfacing rare-but-meaningful words like "refund," "unauthorized," or "OTP" — won't be visible until you either widen the vocabulary or remove Hindi/English filler words first: